## 00 Reduce Raw

Notebook này là **nơi duy nhất đọc file gốc** `data/raw/games.csv`. Nhiệm vụ:

1. Kiểm tra cấu trúc file gốc và tìm ra lỗi header.
2. Sửa header, giữ lại các cột phục vụ câu hỏi nghiên cứu và lưu ra `data/processed/games_reduced.csv`.
3. Giải thích lý do bỏ từng cột.

File rút gọn giữ **nguyên mọi dòng** và **không sửa giá trị nào** của file gốc, chỉ bỏ bớt cột. Các bước audit và làm sạch tiếp theo (notebook 01) dùng file rút gọn này.

### 1. Kiểm tra cấu trúc file gốc

In [1]:
import csv

with open("../data/raw/games.csv", encoding="utf-8-sig", newline="") as f:
    r = csv.reader(f)
    header = next(r)
    row = next(r)

print("Số tên cột:", len(header), "| Số giá trị ở dòng đầu:", len(row))
for h, v in zip(header[:12], row[:12]):
    print(f"{h!r:30} -> {v!r}")

Số tên cột: 39 | Số giá trị ở dòng đầu: 40
'AppID'                        -> '2539430'
'Name'                         -> 'Black Dragon Mage Playtest'
'Release date'                 -> 'Aug 1, 2023'
'Estimated owners'             -> '0 - 0'
'Peak CCU'                     -> '0'
'Required age'                 -> '0'
'Price'                        -> '0.0'
'DiscountDLC count'            -> '0'
'About the game'               -> '0'
'Supported languages'          -> ''
'Full audio languages'         -> '[]'
'Reviews'                      -> '[]'


In [2]:
# Đếm số giá trị (trường) trên từng dòng của file gốc
so_truong = {}

with open("../data/raw/games.csv", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        n = len(row)
        so_truong[n] = so_truong.get(n, 0) + 1

print("Số tên cột trong header:", len(header))
print("Số dòng dữ liệu theo số trường:", so_truong)

Số tên cột trong header: 39
Số dòng dữ liệu theo số trường: {40: 125855}


In [3]:
import pandas as pd

# Cách 1: đọc thẳng -> pandas lấy cột đầu tiên làm index nên dữ liệu bị lệch sang trái 1 cột
doc_thang = pd.read_csv("../data/raw/games.csv", nrows=3)
print("Cách 1 - đọc thẳng (cột AppID lại chứa tên game):")
display(doc_thang[["AppID", "Name", "Release date"]])

# Cách 2: index_col=False -> 8 cột đầu đúng, nhưng từ "DiscountDLC count" trở đi vẫn lệch 1 cột
doc_index = pd.read_csv("../data/raw/games.csv", nrows=3, index_col=False)
print("Cách 2 - index_col=False (cột Windows lại chứa email, Positive sai giá trị):")
display(doc_index[["AppID", "Name", "DiscountDLC count", "Windows", "Positive"]])

Cách 1 - đọc thẳng (cột AppID lại chứa tên game):


,AppID,Name,Release date
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000


Cách 2 - index_col=False (cột Windows lại chứa email, Positive sai giá trị):


,AppID,Name,DiscountDLC count,Windows,Positive
0,2539430,Black Dragon Mage Playtest,0,NaN,0
1,496350,Supipara - Chapter 1 Spring Has Come!,65,support@mangagamer.com,0
2,1034400,Mystery Solitaire The Black Raven,0,support@8floor.net,0


Nhận xét:

- Header chỉ có **39** tên cột, nhưng **tất cả** 125.855 dòng dữ liệu đều có **40** giá trị.
- Đọc thẳng bằng pandas thì toàn bộ dữ liệu lệch 1 cột (cột `AppID` chứa tên game). Dùng `index_col=False` chỉ sửa được 8 cột đầu, các cột phía sau vẫn lệch (cột `Windows` chứa email hỗ trợ).
- Nguyên nhân: tên cột `DiscountDLC count` thật ra là **2 cột** `Discount` và `DLC count` bị dính vào nhau. Ở dòng đầu tiên, sau `Price` có 2 giá trị (`0`, `0`) nhưng header chỉ có 1 tên.
- Cách sửa: tách `DiscountDLC count` thành 2 tên cột, khi đó header có 40 tên, khớp với dữ liệu (bước 2).

### 2. Sửa header, chọn cột và lưu file rút gọn

In [4]:
import csv
import pandas as pd
from pathlib import Path

# ==================================================
# 1. Đường dẫn
# Notebook nằm trong thư mục notebooks/, dataset nằm trong data/
# ==================================================
DATA_PATH = Path("../data/raw/games.csv")
OUTPUT_PATH = Path("../data/processed/games_reduced.csv")
ENCODING = "utf-8-sig"  # utf-8-sig đọc được cả file có BOM lẫn không có

# ==================================================
# 2. Đọc header gốc và sửa lỗi header
# File raw của bộ Steam Games thường có header "DiscountDLC count"
# (dính 2 cột "Discount" và "DLC count" làm một). Header có 39 tên
# nhưng mỗi dòng dữ liệu có 40 giá trị, nên pandas hiểu nhầm cột đầu
# là index và đẩy toàn bộ dữ liệu lệch sang trái 1 cột.
# ==================================================
with open(DATA_PATH, "r", encoding=ENCODING, newline="") as f:
    reader = csv.reader(f)
    raw_header = next(reader)
    first_row = next(reader)

fixed_header = []
for col in raw_header:
    if col == "DiscountDLC count":
        fixed_header.extend(["Discount", "DLC count"])
    else:
        fixed_header.append(col)

print("Số cột trong header gốc :", len(raw_header))
print("Số cột trong header sửa :", len(fixed_header))
print("Số giá trị ở dòng dữ liệu đầu tiên:", len(first_row))

if len(fixed_header) != len(first_row):
    raise ValueError(
        "Số tên cột vẫn không khớp số giá trị trong dòng dữ liệu. "
        "Hãy mở file bằng trình soạn thảo văn bản và kiểm tra lại header."
    )

# ==================================================
# 3. Đọc dữ liệu với tên cột đã sửa
# names=fixed_header + header=0: bỏ dòng header cũ, dùng tên cột mới
# ==================================================
df = pd.read_csv(
    DATA_PATH,
    names=fixed_header,
    header=0,
    encoding=ENCODING,
    low_memory=False,
)

print("\nKích thước ban đầu:", df.shape)
print(
    "Dung lượng ban đầu:",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB",
)

# ==================================================
# 4. Kiểm tra dữ liệu có bị lệch cột không
# AppID phải là số, Release date phải trông giống ngày tháng.
# ==================================================
appid_numeric_rate = pd.to_numeric(df["AppID"], errors="coerce").notna().mean()
print(f"\nTỷ lệ AppID là số: {appid_numeric_rate:.2%}")
display_cols = [c for c in ["AppID", "Name", "Release date", "Price"] if c in df.columns]
print(df[display_cols].head())

if appid_numeric_rate < 0.99:
    raise ValueError(
        "Cột AppID không phải số, dữ liệu vẫn đang bị lệch cột. "
        "Hãy kiểm tra lại header và dữ liệu của file raw."
    )

# ==================================================
# 5. Chọn các cột cần thiết cho nghiên cứu
# ==================================================
keep_columns = [
    "AppID",
    "Name",
    "Release date",
    "Estimated owners",
    "Peak CCU",
    "Price",
    "Discount",
    "DLC count",
    "Positive",
    "Negative",
    "User score",
    "Metacritic score",
    "Recommendations",
    "Achievements",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
    "Categories",
    "Genres",
    "Tags",
]

available_columns = [col for col in keep_columns if col in df.columns]
missing_columns = [col for col in keep_columns if col not in df.columns]

df_reduced = df.loc[:, available_columns].copy()

# Đổi AppID thành id để các bước sau dùng df["id"]
if "AppID" in df_reduced.columns and "id" not in df_reduced.columns:
    df_reduced = df_reduced.rename(columns={"AppID": "id"})

# ==================================================
# 6. Báo cáo các cột không được giữ
# ==================================================
removed_columns = [col for col in df.columns if col not in available_columns]

print("\nSố cột ban đầu:", len(df.columns))
print("Số cột được giữ:", len(available_columns))
print("Số cột không giữ:", len(removed_columns))
print("\nCác cột không có trong file:", missing_columns)
print("\nCác cột bị loại khỏi dataset phân tích:")
print(removed_columns)

# ==================================================
# 7. So sánh dung lượng bộ nhớ
# ==================================================
original_memory = df.memory_usage(deep=True).sum()
reduced_memory = df_reduced.memory_usage(deep=True).sum()

print("\nDung lượng DataFrame ban đầu:", round(original_memory / 1024**2, 2), "MB")
print("Dung lượng DataFrame rút gọn:", round(reduced_memory / 1024**2, 2), "MB")

if original_memory > 0:
    reduction_percent = (original_memory - reduced_memory) / original_memory * 100
    print("Mức giảm bộ nhớ:", round(reduction_percent, 2), "%")

# ==================================================
# 8. Lưu file mới, không ghi đè file gốc
# ==================================================
df_reduced.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nĐã lưu dataset rút gọn tại:", OUTPUT_PATH)
print("Kích thước dataset rút gọn:", df_reduced.shape)

Số cột trong header gốc : 39
Số cột trong header sửa : 40
Số giá trị ở dòng dữ liệu đầu tiên: 40



Kích thước ban đầu: (125855, 40)


Dung lượng ban đầu: 576.87 MB

Tỷ lệ AppID là số: 100.00%
     AppID                                   Name  Release date  Price
0  2539430             Black Dragon Mage Playtest   Aug 1, 2023   0.00
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   5.24
2  1034400      Mystery Solitaire The Black Raven   May 6, 2019   4.99
3  3292190            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024   8.99
4  3631080                          Maze Quest VR  Apr 24, 2025   4.99

Số cột ban đầu: 40
Số cột được giữ: 21
Số cột không giữ: 19

Các cột không có trong file: []

Các cột bị loại khỏi dataset phân tích:
['Required age', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic url', 'Score rank', 'Notes', 'Developers', 'Publishers', 'Screenshots', 'Movies']



Dung lượng DataFrame ban đầu: 576.87 MB
Dung lượng DataFrame rút gọn: 75.43 MB
Mức giảm bộ nhớ: 86.92 %



Đã lưu dataset rút gọn tại: ..\data\processed\games_reduced.csv
Kích thước dataset rút gọn: (125855, 21)


### 3. Lý do bỏ các cột

Với mỗi cột bị bỏ, bảng dưới cho biết tỷ lệ ô trống (NaN, chuỗi rỗng hoặc `[]`), một giá trị ví dụ và lý do bỏ.

In [5]:
ly_do = {
    "About the game": "Văn bản mô tả dài, không dùng cho câu hỏi nghiên cứu",
    "Supported languages": "Danh sách ngôn ngữ, không thuộc RQ1/RQ2/RQ5",
    "Full audio languages": "Danh sách ngôn ngữ lồng tiếng, phần lớn trống",
    "Reviews": "Trích dẫn đánh giá của báo chí dạng văn bản, phần lớn trống",
    "Header image": "Link ảnh, không có thông tin để phân tích",
    "Website": "Link website, không có thông tin để phân tích",
    "Support url": "Link hỗ trợ, không có thông tin để phân tích",
    "Support email": "Email hỗ trợ, không có thông tin để phân tích",
    "Required age": "Gần như toàn bộ bằng 0, không thuộc RQ1/RQ2/RQ5",
    "Windows": "Nền tảng hỗ trợ, không thuộc RQ1/RQ2/RQ5 (gần như mọi game đều có Windows)",
    "Mac": "Nền tảng hỗ trợ, không thuộc RQ1/RQ2/RQ5",
    "Linux": "Nền tảng hỗ trợ, không thuộc RQ1/RQ2/RQ5",
    "Metacritic url": "Link Metacritic, đã có cột Metacritic score",
    "Score rank": "Gần như toàn bộ trống",
    "Notes": "Ghi chú nội dung dạng văn bản, phần lớn trống",
    "Developers": "Hơn 70.000 tên khác nhau, quá rời rạc, không thuộc RQ1/RQ2/RQ5",
    "Publishers": "Hơn 60.000 tên khác nhau, quá rời rạc, không thuộc RQ1/RQ2/RQ5",
    "Screenshots": "Danh sách link ảnh",
    "Movies": "Trống hoàn toàn (100%)",
}

bang_bo_cot = []
for col in removed_columns:
    gia_tri = df[col]
    trong = gia_tri.isna() | gia_tri.astype(str).str.strip().isin(["", "[]"])
    vi_du = gia_tri[~trong].astype(str).head(1).tolist()
    bang_bo_cot.append({
        "Cột": col,
        "% ô trống": round(trong.mean() * 100, 1),
        "Số giá trị khác nhau": gia_tri.nunique(),
        "Ví dụ": vi_du[0][:60] if vi_du else "",
        "Lý do bỏ": ly_do.get(col, ""),
    })

bang_bo_cot = pd.DataFrame(bang_bo_cot)
display(bang_bo_cot)

print("Required age bằng 0:", round((df["Required age"] == 0).mean() * 100, 1), "%")
print("Game hỗ trợ Windows:", round(df["Windows"].mean() * 100, 2), "%")
print("Score rank trống:", round(df["Score rank"].isna().mean() * 100, 2), "%")

,Cột,% ô trống,Số giá trị khác nhau,Ví dụ,Lý do bỏ
0,Required age,0.0,15,0,"Gần như toàn bộ bằng 0, không thuộc RQ1/RQ2/RQ5"
1,About the game,6.7,116776,"Springtime, April: when the cherry trees come ...","Văn bản mô tả dài, không dùng cho câu hỏi nghi..."
2,Supported languages,6.7,19614,['English'],"Danh sách ngôn ngữ, không thuộc RQ1/RQ2/RQ5"
3,Full audio languages,59.3,3785,['Korean'],"Danh sách ngôn ngữ lồng tiếng, phần lớn trống"
4,Reviews,90.3,11987,“And this is the very reason why I believe Fan...,"Trích dẫn đánh giá của báo chí dạng văn bản, p..."
5,Header image,0.1,125664,https://shared.akamai.steamstatic.com/store_it...,"Link ảnh, không có thông tin để phân tích"
6,Website,59.9,40353,http://mangagamer.org/supipara,"Link website, không có thông tin để phân tích"
7,Support url,56.1,36161,http://mangagamer.com,"Link hỗ trợ, không có thông tin để phân tích"
8,Support email,18.0,62321,support@mangagamer.com,"Email hỗ trợ, không có thông tin để phân tích"
9,Windows,0.0,2,True,"Nền tảng hỗ trợ, không thuộc RQ1/RQ2/RQ5 (gần ..."


Required age bằng 0: 99.0 %
Game hỗ trợ Windows: 99.97 %
Score rank trống: 99.97 %


Nhận xét:

- Các cột bị bỏ thuộc 3 nhóm: **link/email/ảnh/video** (không có thông tin để phân tích), **văn bản dài hoặc phần lớn trống** (About the game, Reviews, Notes, Score rank), và **các cột không phục vụ RQ1, RQ2, RQ5** (Required age, Windows/Mac/Linux, ngôn ngữ, Developers/Publishers).
- 21 cột được giữ lại đủ để trả lời 3 câu hỏi nghiên cứu: độ phổ biến (Estimated owners, Peak CCU, Recommendations), đánh giá (Positive, Negative, User score, Metacritic score), giá (Price, Discount, DLC count), đặc điểm game (Genres, Categories, Tags, Achievements, playtime, Release date).
- `User score` vẫn được giữ dù gần như toàn bộ bằng 0, để notebook 01 ghi nhận vấn đề này trong audit.